# Issue #16 — Failure Severity Diagnostics (NFCorpus)

**Research question:** As a failure affects a larger fraction of the system, does retrieval nDCG@10 fall more? Do intended harmless controls stay within an agreed margin of zero damage?

- Source: the benchmark's `ResultRow` parquet, **not** retrieval model output or fabricated measurements. Nothing here reruns the benchmark.
- Only **NFCorpus** is analyzed here; SciFact and FiQA remain held out.
- Multiple detectors create duplicate labels for the *same* query window. The analysis first reduces these to one row per `(run_id, failure, severity, seed, trial)`.
- **Intervals** are built from seed-level values (trials inside a seed overlap). `ci_low`/`ci_high` are a t interval on the seed means; `boot_ci_*` are a percentile bootstrap over seeds, kept for comparison. Neither is known to have nominal coverage with this few seeds, and a wider interval is not automatically a better calibrated one. Both measure **variation across random seeds on the fixed NFCorpus queries, not generalization to new queries.**
- **Trends** are classified from the interval (`increasing` / `decreasing` / `inconclusive`), not from the sign of the mean. A decrease is a finding about the failure, not automatically an injector bug.
- **Severity 1.0** (the whole index replaced, nothing mixed) stays in the tables and figures but is reported separately from the 0.1–0.9 monotonicity check.
- **Controls** are judged against an explicit equivalence margin that you set below. It does not have to equal the runner's `min_drop`.

**To use:** put the full dev run at `results/dev_full.parquet` (it is git-ignored; ask Srikar for run `ca7f8c33c90d` from #79), then run all cells. The notebook finds the repo root itself, so it works whether Jupyter starts in the repo root or in `experiments/`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display


def repo_root(start=None):
    # works from the repo root, from experiments/, or from any folder below the repo
    here = Path(start or Path.cwd()).resolve()
    for folder in (here, *here.parents):
        if (folder / "configs").is_dir() and (folder / "src" / "silent_rag").is_dir():
            return folder
    raise FileNotFoundError(f"No repo root (configs/ and src/silent_rag/) at or above {here}")


ROOT = repo_root()
os.chdir(ROOT)  # so results/ resolves the same way it does for `python -m silent_rag.run`

from silent_rag.eval.severity_analysis import (
    WINDOW_KEY, adjacent_severity_trends, control_validation, interval_note,
    monotonicity_report, plot_severity_curves, prepare_windows, severity_summary,
)

RESULT_PATH = ROOT / "results" / "dev_full.parquet"
OUT_DIR = ROOT / "results" / "severity_analysis"
FIGURE_DIR = OUT_DIR / "figures"
N_BOOT = 2000
BOOTSTRAP_SEED = 20261029
ALPHA = 0.05

# Equivalence margin for the negative controls, in absolute nDCG@10. Deliberately unset:
# it is a team decision and has to be stated with the results. It need not equal min_drop.
CONTROL_MARGIN = None

# Controls whose nDCG drop is 0 by design (the index is untouched, each question gets its
# clean result). They are listed, not tested: a zero here is not a measurement.
ZERO_BY_CONSTRUCTION = ["H_query_topic_shift"]

print("repo root:", ROOT)
print("results:  ", RESULT_PATH, "(found)" if RESULT_PATH.exists() else "(missing)")

In [ ]:
if not RESULT_PATH.exists():
    print(f"No benchmark results at {RESULT_PATH}")
    print("Nothing below will run, and no numbers are reported, until the parquet is there.")
    raw = windows = None
else:
    raw = pd.read_parquet(RESULT_PATH)
    windows = prepare_windows(raw)
    print(f"Loaded {len(raw)} ResultRows -> {len(windows)} unique experiment windows")
    display(windows.groupby(["failure", "severity"], sort=True).agg(
        seeds=("seed", "nunique"), windows=("trial", "size")
    ).reset_index())

## Is this the run from #79?

A read-only check that the file is the full dev run, before trusting anything below. The expected values are quoted from the description of PR #79; they are not computed here. The harmful-rate table re-thresholds `p_value` and `ndcg_drop` the way #79 did, so it can be compared with that PR's table by eye (the stored `harmful` column is at the config's `min_drop`, which is 0 for `dev_full.yaml`).

In [ ]:
EXPECTED_79 = {"run_id": "ca7f8c33c90d", "rows": 100_800, "windows": 7_200, "conditions": 36, "seeds": 10}

if windows is not None:
    found = {
        "run_id": raw["run_id"].iloc[0],
        "rows": len(raw),
        "windows": len(windows),
        "conditions": windows.groupby(["failure", "severity"]).ngroups,
        "seeds": windows["seed"].nunique(),
    }
    check = pd.DataFrame({"expected (#79)": EXPECTED_79, "found": found})
    check["match"] = check["expected (#79)"] == check["found"]
    display(check)
    if not check["match"].all():
        print("This is not the #79 run (or not all of it). The analysis still runs; compare with #79 with care.")

    per_window = raw.drop_duplicates(list(WINDOW_KEY))
    rates = {}
    for min_drop in (0.0, 0.02, 0.06):
        hit = (per_window["p_value"] < ALPHA) & (per_window["ndcg_drop"] >= min_drop)
        rates[f"min_drop {min_drop:g}"] = hit.groupby([per_window["failure"], per_window["severity"]]).mean()
    print("Share of windows harmful (p < 0.05 and drop >= min_drop), to compare with #79's table:")
    display(pd.DataFrame(rates).round(2))

In [ ]:
if windows is not None:
    summary = severity_summary(windows, n_boot=N_BOOT, alpha=ALPHA, seed=BOOTSTRAP_SEED)
    note = interval_note(summary)
    if note:
        print("NOTE:", note)
    display(summary)
    paths = plot_severity_curves(summary, FIGURE_DIR)
    print("Figures:")
    for path in paths:
        print(" ", path)
    # Save analysis tables alongside the figures; no held-out evaluation.
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUT_DIR / "severity_summary.csv", index=False)

## Does damage grow with severity?

`trend` is read from the t interval of the seed-level paired differences; `boot_trend` is the same call with the bootstrap interval. Where the two disagree, treat the step as fragile. `scope` separates the 0.1–0.9 range from the step into severity 1.0, and the report below judges only the range, with the endpoint next to it.

No multiplicity correction is applied across steps.

In [ ]:
if windows is not None:
    trends = adjacent_severity_trends(windows, n_boot=N_BOOT, alpha=ALPHA, seed=BOOTSTRAP_SEED)
    display(trends)
    trends.to_csv(OUT_DIR / "adjacent_severity_trends.csv", index=False)

    disagree = trends.loc[trends["trend"] != trends["boot_trend"]]
    print(f"Steps where the t interval and the bootstrap disagree: {len(disagree)}")
    if len(disagree):
        display(disagree)

    report = monotonicity_report(trends)
    display(report)
    report.to_csv(OUT_DIR / "monotonicity_report.csv", index=False)
    print("range_verdict covers the steps below severity 1.0 only; endpoint_trend is the step into 1.0.")
    print("'non_monotonic' and a 'decreasing' endpoint describe the measurements. They are not, by themselves, injector bugs.")

## Do the controls stay harmless?

Each control is judged per severity against `CONTROL_MARGIN`, so a control that runs at a single severity (the full swaps) is covered here even though it has no trend row.

- `equivalent`: the drop is within ±margin.
- `not_worse`: the one-sided upper bound is below the margin, so degradation of that size is ruled out, but the drop is not shown to be within −margin (nDCG improved, or the seeds are too noisy on that side). Not shown flat.
- `exceeds_margin`: a failed control.
- `inconclusive`: the interval crosses the margin. Degradation is neither shown nor ruled out; this is **not** evidence that the control is harmless.
- `not_tested`: zero by construction (`evidence = by_construction`), or an exact zero nobody declared (`all_zero_unexplained`). Neither counts as a control that passed.

In [ ]:
if windows is not None:
    if CONTROL_MARGIN is None:
        print("CONTROL_MARGIN is not set, so the controls are not validated.")
        print("Agree a margin with the team, set it in the first cell, and say what it is wherever these results are reported.")
        display(summary.loc[~summary["expected_harmful"] & (summary["failure"] != "clean")])
    else:
        present = [f for f in ZERO_BY_CONSTRUCTION if f in set(windows["failure"])]
        controls = control_validation(windows, margin=CONTROL_MARGIN, by_construction=present, alpha=ALPHA)
        display(controls)
        controls.to_csv(OUT_DIR / "control_validation.csv", index=False)
        print(f"Margin: {CONTROL_MARGIN} absolute nDCG@10 (chosen above, not derived from the data).")

## Interpretation / follow-up checklist

- For **harmful failures**, read `range_verdict` for the 0.1–0.9 question in #16 and `endpoint_trend` separately. A step is only a decrease when its interval is below zero. Take a `non_monotonic` range or a `decreasing` endpoint to the injector owner as an observation with the seed/trial examples, and decide together whether it is how the failure behaves or something to fix.
- For **negative controls**, report the margin with the verdict. `not_worse` and `equivalent` are different statements, and `not_tested` controls are not evidence that anything was harmless.
- **Intervals** cover seed-to-seed variation on the fixed NFCorpus queries only. Do not present them as uncertainty about new queries or other datasets.
- If intervals are missing, the run had one seed: rerun with `seeds: [0, 1, 2, ...]`.
- **Do not** run the held-out final config to decide thresholds, margins, severity definitions or analysis choices.